In [ ]:
import ee
import folium
import geemap
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='tejeshn3')
file_path = '/content/drive/MyDrive/GEE/Modified_Sorghum_BlockReport_1975-1980.xlsx'
df = pd.read_excel(file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import ipywidgets as widgets
import geemap
import pandas as pd
import math
import numpy as np
from geopy.distance import geodesic
from shapely.geometry import box
from geopy import Point
from IPython.display import display, HTML
import ee
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from google.colab import output

# === STYLING ===
display(HTML("""
<style>
    .custom-year-bar {
        position: absolute;
        bottom: 25px;
        left: 50%;
        transform: translateX(-50%);
        display: flex;
        gap: 4px;
        padding: 6px 10px;
        background: rgba(255, 255, 255, 0.9);
        border-radius: 12px;
        box-shadow: 0px 2px 10px rgba(0,0,0,0.25);
        z-index: 999;
        overflow-x: auto;
        white-space: nowrap;
        scrollbar-width: none;
    }
    .custom-year-bar::-webkit-scrollbar {
        display: none;
    }
    .year-button {
        font-size: 14px;
        padding: 4px 10px;
        border: none;
        background-color: transparent;
        border-radius: 6px;
        color: #333;
        cursor: pointer;
        transition: all 0.2s ease;
    }
    .year-button:hover {
        background-color: #e0e0e0;
    }
    .year-button.active {
        background-color: #4285f4;
        color: white;
        font-weight: bold;
    }

    .widget-label,
    .widget-readout,
    .widget-inline-hbox input,
    .widget-dropdown,
    .widget-slider {
        color: black !important;
    }

    .leaflet-control-layers,
    .leaflet-control-attribution,
    .leaflet-draw,
    .leaflet-control-geocoder,
    .leaflet-top.leaflet-right {
        display: none !important;
    }


    /* .leaflet-control-zoom { display: none !important; } */
    /* .leaflet-control-scale { display: none !important; } */
</style>
"""))

# === DATA PREP ===
lat_col = "Center Lat"
lon_col = "Center Long"
df['Year'] = pd.to_datetime(df['Date']).dt.year

parameters = {
    "Yield": "Unadjusted Yield (kg/ha))",
    "Water Productivity": "Yield/App Irrigation Prod(kg/ha/mm)",
    "Nitrogen Productivity": "Yield/App Nitrogen Prod(kg/kg)"
}

bbox_area = 615040000  # ≈ 24.8 km x 24.8 km

def create_bounding_box(point, area_m2):
    len_km = math.sqrt(float(area_m2)) / 1000
    hyp = math.sqrt(2 * (len_km ** 2))
    half_hyp = hyp / 2
    ne_point = geodesic(kilometers=half_hyp).destination(point, bearing=45)
    sw_point = geodesic(kilometers=half_hyp).destination(point, bearing=225)
    return box(sw_point.longitude, sw_point.latitude, ne_point.longitude, ne_point.latitude)

def get_color(value, min_val, max_val):
    norm = mcolors.Normalize(vmin=min_val, vmax=max_val)
    cmap = mcolors.LinearSegmentedColormap.from_list("orange_to_green", ["#ff8c00", "#006400"])
    rgba = cmap(norm(value))
    return mcolors.to_hex(rgba)

# === CLEAN MAP SETUP ===
m = geemap.Map(
    layers_control=False,
    draw_control=False,
    fullscreen_control=False,
    toolbar_control=False,
    attribution_control=False
)
m.add_basemap("Esri.WorldImagery")
m.setCenter(77.6332, 11.0371, 7)

year_min, year_max = df['Year'].min(), df['Year'].max()
current_year = year_min

# === Tamil Nadu boundary ===
tamil_nadu = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "India"),
        ee.Filter.eq("ADM1_NAME", "Tamil Nadu")
    )
)

# === CALLBACK FUNCTION ===
def on_year_click(selected_year):
    global current_year
    current_year = int(selected_year)
    update_map(param_dropdown.value, current_year)

    js = f"""
    setTimeout(() => {{
        window.activeYear = {current_year};
        const all = document.querySelectorAll('.year-button');
        all.forEach(btn => {{
            if (!btn.classList.contains('year-button')) return;
            btn.classList.remove('active');
        }});
        const active = document.getElementById("year-{current_year}");
        if (active) {{
            active.classList.add("active");
        }}
        const bar = document.getElementById("year-bar");
        if (bar && active) {{
            const barRect = bar.getBoundingClientRect();
            const btnRect = active.getBoundingClientRect();
            const offset = btnRect.left - barRect.left - (barRect.width / 2) + (btnRect.width / 2);
            bar.scrollBy({{ left: offset, behavior: 'smooth' }});
        }}
    }}, 0);
    """
    display(HTML(f"<script>{js}</script>"))

output.register_callback('select_year', on_year_click)

def build_year_buttons():
    buttons_html = "<button id='playPauseBtn' class='year-button' onclick='togglePlayPause()'>▶</button>"
    for yr in range(year_min, year_max + 1):
        class_name = "year-button active" if yr == current_year else "year-button"
        buttons_html += f"<button id='year-{yr}' class='{class_name}' onclick=\"google.colab.kernel.invokeFunction('select_year', ['{yr}'], {{}})\">{yr}</button>"

    html = f"""
    <div class='custom-year-bar' id='year-bar'>{buttons_html}</div>
    <script>
    window.activeYear = {current_year};
    let playing = false;
    let playInterval;
    function togglePlayPause() {{
        const btn = document.getElementById("playPauseBtn");
        playing = !playing;
        btn.textContent = playing ? "⏸" : "▶";
        if (playing) {{
            playInterval = setInterval(() => {{
                let next = window.activeYear + 1;
                if (next > {year_max}) next = {year_min};
                window.activeYear = next;
                google.colab.kernel.invokeFunction('select_year', [String(next)], {{}});
            }}, 10000);
        }} else {{
            clearInterval(playInterval);
        }}
    }}
    const bar = document.getElementById("year-bar");
    const active = document.querySelector(".year-button.active");
    if (bar && active) {{
        const barRect = bar.getBoundingClientRect();
        const btnRect = active.getBoundingClientRect();
        const offset = btnRect.left - barRect.left - (barRect.width / 2) + (btnRect.width / 2);
        bar.scrollBy({{ left: offset, behavior: 'smooth' }});
    }}
    </script>
    """
    display(HTML(html))

# === MAP UPDATE ===
def update_map(param_name, selected_year):
    param_col = parameters[param_name]
    df_filtered = df[df['Year'] == selected_year].groupby([lat_col, lon_col])[param_col].mean().reset_index()
    if df_filtered.empty:
        return

    min_val = df_filtered[param_col].min()
    max_val = df_filtered[param_col].max()
    palette = [get_color(v, min_val, max_val) for v in np.linspace(min_val, max_val, 10)]

    features = []
    for _, row in df_filtered.iterrows():
        lon, lat, value = row[lon_col], row[lat_col], row[param_col]
        color_hex = get_color(value, min_val, max_val)
        point = Point(lat, lon)
        bbox = create_bounding_box(point, bbox_area)
        ee_geom = ee.Geometry.Polygon(list(bbox.exterior.coords))
        features.append(ee.Feature(ee_geom, {param_name: value, "color": color_hex}))

    fc = ee.FeatureCollection(features)
    clipped_fc = fc.map(lambda f: f.setGeometry(f.geometry().intersection(tamil_nadu.geometry(), 1)))
    styled_fc = clipped_fc.map(lambda f: f.set({'style': {'color': f.get('color'), 'width': 1}}))
    styled_image = styled_fc.style(styleProperty='style')

    for lyr in list(m.layers):
        if hasattr(lyr, 'name') and lyr.name == "Active Layer":
            m.remove_layer(lyr)

    m.addLayer(styled_image, {}, "Active Layer", opacity=opacity_slider.value)

    tn_outline = tamil_nadu.style(color='black', fillColor='00000000', width=2)
    m.addLayer(tn_outline, {}, "Tamil Nadu Boundary")

    m.colorbar = None
    m.remove_colorbar()
    m.add_colorbar(
        vis_params={'min': min_val, 'max': max_val, 'palette': palette},
        label=f"{param_name} Scale"
    )

# === UI WIDGETS ===
param_dropdown = widgets.Dropdown(
    options=parameters.keys(),
    value="Yield",
    description="Parameter:"
)

opacity_slider = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Opacity:',
    continuous_update=False
)

param_dropdown.observe(lambda c: update_map(c["new"], current_year), names='value')
opacity_slider.observe(lambda c: update_map(param_dropdown.value, current_year), names='value')

# === FINAL INIT ===
m.add_widget(param_dropdown)
m.add_widget(opacity_slider)
build_year_buttons()
update_map(param_dropdown.value, current_year)
display(m)


Map(center=[11.0371, 77.6332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…